# Logistic regression — from scratch

*Companion to **Eps 24–28** of the Logistic Regression series.*

Same five-line training loop as linear regression. Two substitutions: a sigmoid in the forward pass, binary cross-entropy in place of MSE. By the end you'll have trained a classifier three ways.

## Chapter 1 · Setup

## Chapter 2 · The dataset

Fake spam dataset. Two features per email:

- `x1` — number of suspicious words ("click here", "free", "urgent")
- `x2` — capitalization ratio (fraction of letters in CAPS)

Label is 1 if spam, 0 if not. Real spam classifiers use thousands of features. Two is enough to **see** what's going on.

Sanity check.

Plot it.

Two clusters. Some overlap in the middle. **A line through that middle would separate most of them.** Finding that line is what we're going to do.

## Chapter 3 · Try linear regression first — watch it break

Labels are 0 and 1. Just numbers. Linear regression fits any numbers. Let's plug it in and see what happens.

Predict for a few example emails.

Look at those numbers.

- Email 1: prediction `-0.14`. *Negative.* What does negative probability mean? Nothing.
- Email 4: prediction `+2.06`. *More than 1.* Probability above 100%? Doesn't exist.

**Linear regression has no idea our output is supposed to be a probability.** We need a model that knows.

## Chapter 4 · The sigmoid function

The sigmoid takes any real number and squashes it into `(0, 1)`. Always. No matter how huge or tiny the input.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Worked example: what's `σ(0)`?

Exactly 0.5. The middle.

**Worked example:** what's `σ(1)`?

About 0.73. Mildly positive.

**Worked example:** what about a big positive input?

About 0.99. Confidently positive — but not 1.0.

**Worked example:** big negative input?

About 0.0067. Confidently negative — but not 0.

**The asymptotes never reach.** Push the input to 100, the output gets really close to 1, but never exactly. That's the feature, not a bug — it keeps the math well-behaved.

Plot the full S-curve.

## Chapter 5 · Logistic regression = linear + sigmoid

Logistic regression is two steps:

1. **Linear part.** `z = b + w₁·x₁ + w₂·x₂ + …` Just like linear regression. Output is any real number.
2. **Sigmoid.** Squash `z` through the sigmoid. Output is in `(0, 1)`. That's the predicted probability.

$$\hat{y} = \sigma(b + w_1 x_1 + w_2 x_2)$$

Worked example. Bias `b=1`, weights `w₁=2, w₂=-1`. New email features `x₁=3, x₂=2`.

Then squash.

About 95%. The model says: *given those features and those weights, this email has a 95% chance of being spam.*

Now — where do good weights come from? Training.

## Chapter 6 · Log loss

Before we train, we need a loss function. MSE doesn't work here — sigmoid's flat tails make the gradient vanish when the model is confidently wrong. (We'll see it in a moment.)

The right loss for binary classification is **log loss** (also called **binary cross-entropy**):

$$L = -\frac{1}{N}\sum_i \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

The formula has two cases packed into one. Let me unpack.

**Case 1: true label is 1.** The `(1 - yᵢ)` term is zero — drops out. Only `−log(ŷᵢ)` remains.

**Case 2: true label is 0.** The `yᵢ` term drops out. Only `−log(1 − ŷᵢ)` remains.

One formula, two cases. The case you're in depends on the truth.

**Worked example.** Truth is 1, model predicts 0.92. Loss?

Tiny. Model was right and confident.

**Worked example.** Truth is 1, model predicts 0.5 — uncertain.

0.69. Moderate punishment for being uncertain.

**Worked example.** Truth is 1, model predicts 0.01 — confidently wrong.

4.6. Big. **The loss explodes as you become more confidently wrong.** That's what we want — strong gradient signal exactly when the model needs to learn most.

Plot both curves.

Asymptote on the wrong side of each curve. Loss is small when the prediction matches the truth, blows up when it's confidently wrong.

## Chapter 7 · Train it by hand

Three parameters: `w₁`, `w₂`, `b`. Gradient descent is the same loop as linear regression, with sigmoid in the forward pass and log loss as the criterion.

Forward pass — compute predicted probabilities for the whole batch.

Gradient. (For log loss + sigmoid, the math is unusually clean — the gradients work out to `prediction − truth`, no chain-rule mess.)

Initialize parameters.

One step.

Loss went down. Now run the full loop.

Plot the loss curve.

## Chapter 8 · The decision boundary

Plot the trained model's *decision boundary*. For logistic regression, the boundary is the line where predicted probability equals 0.5 — equivalently, where `W·x + b = 0`.

Yellow line splits spam from not-spam. **A handful of points are on the wrong side** — those are the ambiguous emails the model isn't sure about. Tightening the threshold (next series, classification) is how you trade precision for recall.

## Chapter 9 · The same thing in PyTorch — 5-line loop, two changes from LR

Same five-line loop from the linear regression notebook. Two substitutions:

- Add a sigmoid in the forward pass.
- Use binary cross-entropy as the loss.

PyTorch's `nn.BCEWithLogitsLoss` combines sigmoid + BCE in one call (more numerically stable).

Set up model + optimizer + loss.

The five lines.

Compare to the by-hand version.

Same answer. **Three lines of the five-line loop are literally identical to linear regression.** Two changes give you a classifier.

## Chapter 10 · The same thing in sklearn — 3 lines

Read out the coefficients.

Sklearn's coefficients differ from ours because **sklearn defaults to L2 regularization** (shrinks weights toward zero to prevent overfitting). Our by-hand version had no regularization. Both predict the same boundary direction; sklearn's weights are just smaller in magnitude.

Predict probabilities for a few example emails.

Real probabilities now. All between 0 and 1. **That's what we built sigmoid for.**

## Chapter 11 · Read the trained model

Coefficients in logistic regression have a clean interpretation — they're effects on the **log-odds**. Exponentiating a coefficient gives the multiplicative effect on the odds.

Exponentiate to get the multiplicative effects.

**That's a real model you can interpret.** Each feature has a clear, multiplicative effect on the odds of spam. Not a black box.

## Chapter 12 · What we did

1. Built a synthetic spam dataset (two features, two classes).
2. Tried linear regression — saw it produce probabilities below 0 and above 1. Broken.
3. Introduced the sigmoid function. Worked through five numeric examples to feel it.
4. Combined linear + sigmoid into logistic regression. Worked example.
5. Introduced log loss. Three worked examples to feel why it's the right loss.
6. Trained the model by hand. One step, then the full loop.
7. Plotted the decision boundary.
8. Same model in PyTorch (5 lines, two substitutions from LR) and sklearn (3 lines).
9. Read the trained coefficients as multiplicative effects on the odds.

**Next series: classification.** Threshold. Confusion matrix. Precision, recall, ROC. How to know whether your classifier is actually good.